# Using OpenAI Bedrock models with Strands Agent

## Overview

Strands Agents is an SDK that takes a model-driven approach to building and running AI agents in just a few lines of code. Strands supports multiple providers and models hosted anywhere.

In this example, we will show you how to use `gpt-5.4` model hosted in Amazon Bedrock as the underlying model in your Strands Agent. We will use a simple agent use case with a weather and a get time tool.

We will use the OpenAI Responses API.


## Agent Details
<div style="float: left; margin-right: 20px;">
    
|Feature             |Description                                        |
|--------------------|---------------------------------------------------|
|Feature used        |OpenAI model                                      |
|Agent Structure     |Single agent architecture                          |

</div>

## Architecture

<div style="text-align:center">
    <img src="images/simple_agent.png" width="65%" />
</div>

## Key Features
* **OpenAI model**: using a model provided via OpenAI on Amazon Bedrock
* **OpenAI Responses API**: using the Responses API

## Setup and prerequisites

### Prerequisites
* Python 3.10+
* AWS Account

Install the required packages for our Strands Agent:

In [ ]:
# installing pre-requisites
!pip install -r requirements.txt

### Importing dependency packages

Now let's import the dependency packages:

In [4]:
import os
from datetime import datetime
from datetime import timezone as tz
from typing import Any
from zoneinfo import ZoneInfo

from strands import Agent, tool
from strands.models.openai_responses import OpenAIResponsesModel


### OpenAI keys

Let's now set up the API Keys. You have two choices: generate a short-term Bedrock API key from the AWS Console, or use AWS credentials.

In [ ]:
os.environ["AWS_BEARER_TOKEN_BEDROCK"] = "<YOUR_BEDROCK_API_KEY>"

os.environ["AWS_REGION"] = "<AWS_REGION>"
os.environ["AWS_ACCESS_KEY_ID"] = "<YOUR_AWS_ACCESS_KEY_ID>"
os.environ["AWS_SECRET_ACCESS_KEY"] = "<YOUR_AWS_SECRET_ACCESS_KEY>"
os.environ["AWS_SESSION_TOKEN"] = "<YOUR_AWS_SESSION_TOKEN>"

region = os.environ["AWS_REGION"]

# uncomment next line to test:
#!aws sts get-caller-identity

### Setting up custom tools

Let's now set up two tools to test our agent:

In [6]:
@tool
def current_time(timezone: str = "UTC") -> str:
    if timezone.upper() == "UTC":
        timezone_obj: Any = tz.utc
    else:
        timezone_obj = ZoneInfo(timezone)

    return datetime.now(timezone_obj).isoformat()


@tool
def current_weather(city: str) -> str:
    # Dummy implementation. Replace with actual weather API call.
    return "sunny"

### Test connection to Bedrock

If you are using a Bedrock API key, run the next cell, otherwise skip.

In [8]:
api_key = os.environ["AWS_BEARER_TOKEN_BEDROCK"]

model = OpenAIResponsesModel(
    model_id="openai.gpt-5.4",
    client_args={
        "api_key": api_key,
        "base_url": f"https://bedrock-mantle.{region}.api.aws/openai/v1",
    },
)

agent = Agent(model=model)
response = agent("What is the capital of British Columbia?")

Victoria.

### Try a Bedrock-generated token

In [9]:
from aws_bedrock_token_generator import provide_token

model = OpenAIResponsesModel(
    model_id="openai.gpt-5.4",
    client_args={
        "api_key": provide_token(region=region),
        "base_url": f"https://bedrock-mantle.{region}.api.aws/openai/v1",
    },
)

system_prompt = "You are a simple agent that can tell the time and the weather"

agent = Agent(
    model=model,
    system_prompt=system_prompt,
    tools=[current_time, current_weather],
)

response = agent("What is the time and weather in Philadelphia?")



Tool #1: current_time

Tool #2: current_weather
In Philadelphia, it’s currently 5:13 PM local time, and the weather is sunny.

Next we can take a look at the usage of our agent for the last query by examining the response `metrics`:

In [ ]:
from pprint import pprint

pprint(vars(response.metrics))


{'accumulated_metrics': {'latencyMs': 0},
 'accumulated_usage': {'inputTokens': 284,
                       'outputTokens': 76,
                       'totalTokens': 360},
 'agent_invocations': [AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='f265095a-7206-4416-b812-fa6ca3f2710d',
                                                                    usage={'inputTokens': 96,
                                                                           'outputTokens': 52,
                                                                           'totalTokens': 148}),
                                               EventLoopCycleMetric(event_loop_cycle_id='04223e1d-4a02-43bb-a77c-8e90be7bd24e',
                                                                    usage={'inputTokens': 188,
                                                                           'outputTokens': 24,
                                                                           'totalTokens': 212})]

### Congratulations!

In this notebook you learned how to use Bedrock with OpenAI using the Responses API. Try other tools in your environment.